In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys
import numpy as np
import torch

PROJECT_ROOT = "/content/drive/MyDrive/mind-recommender"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

PROJECT_ROOT: /content/drive/MyDrive/mind-recommender
RESULTS_DIR: /content/drive/MyDrive/mind-recommender/results


Phase 4 — Model Architecture & Implementation



In [3]:
embedding_matrix = np.load(os.path.join(RESULTS_DIR, "embedding_matrix.npy"))
train_samples = np.load(os.path.join(RESULTS_DIR, "train_samples.npy"), allow_pickle=True).tolist()
dev_samples = np.load(os.path.join(RESULTS_DIR, "dev_samples.npy"), allow_pickle=True).tolist()

print("Embedding matrix shape:", embedding_matrix.shape)
print("Train samples:", len(train_samples))
print("Dev samples:", len(dev_samples))

Embedding matrix shape: (20774, 300)
Train samples: 214962
Dev samples: 104697


In [4]:
from src.model import NRMSModel

print("NRMSModel imported successfully")

NRMSModel imported successfully


In [5]:
sample = train_samples[0]

print("History length:", len(sample["history"]))
print("Num candidates:", len(sample["candidates"]))
print("Labels:", sample["labels"])

History length: 50
Num candidates: 5
Labels: [1, 0, 0, 0, 0]


In [6]:
history = torch.tensor(sample["history"], dtype=torch.long).unsqueeze(0)       # (1, hist_len, title_len)
candidates = torch.tensor(sample["candidates"], dtype=torch.long).unsqueeze(0) # (1, num_candidates, title_len)
hist_mask = (history.abs().sum(dim=-1) != 0).long()

print("History shape:", history.shape)
print("Candidates shape:", candidates.shape)
print("Hist mask shape:", hist_mask.shape)

History shape: torch.Size([1, 50, 30])
Candidates shape: torch.Size([1, 5, 30])
Hist mask shape: torch.Size([1, 50])


In [7]:
model = NRMSModel(
    embedding_matrix=embedding_matrix,
    num_heads=16,
    head_dim=16,
    dropout=0.2
)

print(model)

NRMSModel(
  (news_encoder): NewsEncoder(
    (word_embed): Embedding(20774, 300, padding_idx=0)
    (dropout): Dropout(p=0.2, inplace=False)
    (proj): Linear(in_features=300, out_features=256, bias=True)
    (multihead_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (additive_attn): AdditiveAttention(
      (proj): Linear(in_features=256, out_features=200, bias=True)
      (query): Linear(in_features=200, out_features=1, bias=False)
    )
  )
  (user_encoder): UserEncoder(
    (multihead_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (additive_attn): AdditiveAttention(
      (proj): Linear(in_features=256, out_features=200, bias=True)
      (query): Linear(in_features=200, out_features=1, bias=False)
    )
    (dropout): Dropout(p=0.2, inplace=False)
  )
)


In [8]:
with torch.no_grad():
    scores = model(history, candidates, hist_mask)

print("Scores shape:", scores.shape)
print("Scores:", scores)
print("Any NaN in scores?", torch.isnan(scores).any().item())

Scores shape: torch.Size([1, 5])
Scores: tensor([[-0.0019,  0.0002, -0.0019, -0.0009, -0.0015]])
Any NaN in scores? False


Phase 5 — Training & Hyperparameter Tuning


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [10]:
from torch.utils.data import DataLoader
from src.data_loader import NewsDataset, collate_fn

train_dataset = NewsDataset(train_samples)
dev_dataset = NewsDataset(dev_samples)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

print("DataLoaders ready ✅")

DataLoaders ready ✅


In [11]:
import os

os.makedirs("models", exist_ok=True)
print("models folder ready ✅")

models folder ready ✅


Phase 6 — Evaluation & Analysis

In [13]:
from src.train import train_model

model, losses = train_model(
    train_loader=train_loader,
    val_loader=dev_loader,
    embedding_matrix=embedding_matrix,
    device=device,
    epochs=5
)

Epoch 1: 100%|██████████| 3359/3359 [11:32<00:00,  4.85it/s, loss=1.39]



Epoch 1: Loss = 1.4825
Running evaluation...

Evaluation Results:
AUC: 0.6295
MRR: 0.5559
nDCG@5: 0.6660
nDCG@10: 0.6660


Epoch 2: 100%|██████████| 3359/3359 [11:40<00:00,  4.79it/s, loss=1.41]



Epoch 2: Loss = 1.4126
Running evaluation...

Evaluation Results:
AUC: 0.6394
MRR: 0.5673
nDCG@5: 0.6746
nDCG@10: 0.6746


Epoch 3: 100%|██████████| 3359/3359 [11:39<00:00,  4.80it/s, loss=1.35]



Epoch 3: Loss = 1.3804
Running evaluation...

Evaluation Results:
AUC: 0.6487
MRR: 0.5754
nDCG@5: 0.6808
nDCG@10: 0.6808


Epoch 4: 100%|██████████| 3359/3359 [11:40<00:00,  4.80it/s, loss=1.35]



Epoch 4: Loss = 1.3579
Running evaluation...

Evaluation Results:
AUC: 0.6552
MRR: 0.5834
nDCG@5: 0.6868
nDCG@10: 0.6868


Epoch 5: 100%|██████████| 3359/3359 [11:39<00:00,  4.80it/s, loss=1.35]



Epoch 5: Loss = 1.3420
Running evaluation...

Evaluation Results:
AUC: 0.6562
MRR: 0.5842
nDCG@5: 0.6875
nDCG@10: 0.6875
